# Consciousness Collapse — Reproducibility Notebook

This notebook reproduces the core empirical result in **“Consciousness Collapse: Loss of Consciousness as Impedance Catastrophe”**:

- fit the collapse curve on the condensed cross-domain dataset (43 anchor points)
- **domain-normalize η** before pooling (required for cross-domain comparability)
- estimate **k** and **η₀**
- bootstrap a 95% CI for **k**

## Files expected in `data/`

- `constructed_eta_R_dataset.csv`  (required)  
  Columns expected: `domain`, `eta`, `R` (extra columns are fine)

Optional:
- `Data.pdf` (provenance / source compilation)
- `PERFECT_FINAL_with_Consciousness_Collapse_Addendum.pdf` (paper + clarification)



In [ ]:
# Core dependencies
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt


In [ ]:
# Load dataset
PATH = "data/constructed_eta_R_dataset.csv"
df = pd.read_csv(PATH)

required = {"domain","eta","R"}
missing = required - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

df = df.copy()
df = df.sort_values(["domain","eta"]).reset_index(drop=True)

print("Rows:", len(df))
print("Domains:", sorted(df["domain"].unique()))
print(df.head())


In [ ]:
# Domain-normalize η (z-score within each domain)
df["eta_dn"] = df.groupby("domain")["eta"].transform(lambda x: (x - x.mean()) / x.std())

assert not df["eta_dn"].isna().any(), "eta_dn contains NaNs (check domains with <2 points)"

print("eta_dn range:", float(df["eta_dn"].min()), "to", float(df["eta_dn"].max()))


In [ ]:
# Quick visual sanity check (per-domain)
plt.figure(figsize=(7,4))
for d, g in df.groupby("domain"):
    plt.scatter(g["eta_dn"], g["R"], s=25, label=d)
plt.xlabel("eta_dn (domain-normalized)")
plt.ylabel("R")
plt.title("Per-domain points on normalized η axis")
plt.legend(ncol=2, fontsize=8)
plt.show()


In [ ]:
# Fit pooled logistic collapse on normalized η
eta = df["eta_dn"].to_numpy(float)
R = df["R"].to_numpy(float)

def logistic(x, k, x0):
    # decreasing in x
    return 1.0 / (1.0 + np.exp(k * (x - x0)))

popt, _ = curve_fit(
    logistic,
    eta,
    R,
    p0=[1.5, 0.0],
    bounds=([0.1, -5.0], [10.0, 5.0]),
    maxfev=100000
)

k_hat, eta0_hat = popt
Rhat = logistic(eta, k_hat, eta0_hat)

sse = float(np.sum((R - Rhat)**2))
sst = float(np.sum((R - np.mean(R))**2))
r2  = 1.0 - sse/sst

print("k_hat   =", float(k_hat))
print("eta0_hat=", float(eta0_hat))
print("SSE     =", sse)
print("R^2     =", r2)


In [ ]:
# Overlay plot
xgrid = np.linspace(np.min(eta), np.max(eta), 400)

plt.figure(figsize=(7,4))
plt.scatter(eta, R, s=30, label="data (pooled)")
plt.plot(xgrid, logistic(xgrid, k_hat, eta0_hat), linewidth=2, label=f"fit: k={k_hat:.3f}, eta0={eta0_hat:.3f}")
plt.xlabel("eta_dn (domain-normalized)")
plt.ylabel("R")
plt.title("Pooled logistic collapse fit")
plt.legend()
plt.show()


In [ ]:
# Bootstrap 95% CI for k (deterministic seed)
rng = np.random.default_rng(42)
ks = []
n = len(eta)

for _ in range(2000):
    idx = rng.integers(0, n, size=n)
    try:
        popt_b, _ = curve_fit(
            logistic,
            eta[idx],
            R[idx],
            p0=[k_hat, eta0_hat],
            bounds=([0.1, -5.0], [10.0, 5.0]),
            maxfev=50000
        )
        ks.append(popt_b[0])
    except Exception:
        pass

ks = np.array(ks, dtype=float)
ci = np.percentile(ks, [2.5, 50, 97.5])

print("bootstrap fits:", len(ks))
print("bootstrap k median:", float(ci[1]))
print("k 95% CI:", [float(ci[0]), float(ci[2])])


## Notes on coordinate dependence of k

The steepness parameter **k** rescales under linear transformations of η.

If η' = a η + b, then k' = k / a.

For cross-domain comparisons, always state how η was normalized. The default in this notebook is
**domain-wise z-scoring** prior to pooling.
